# Library breadth

Reproduce `library_breadth_with_opt.pdf`: ridge-optimized risk at two aspect ratios,
followed by optimal training breadth versus test breadth. The default run loads
bundled numerical arrays and redraws the figure without evaluating risk or searching
for optima. Set `RECOMPUTE=True` to calculate the displayed studies from the explicit
controls below. Numerical stages, presentation, and PDF export are separate.

Risk excludes the independent test-noise floor and is shown in units of $\sigma^2$.
The black reference is the analytical small-$r$ optimal breadth $f_*^{(0)}$.
The optimal-breadth search refines detected basins; its tolerance does not guarantee
that every possible basin has been found. Dashed jump connectors use stored
barrier evidence at both adjacent test-breadth values.

In [ ]:
from pathlib import Path
from copy import deepcopy
import importlib
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'tools' / 'portable_results.py').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'tools' / 'portable_results.py').is_file():
    raise RuntimeError('Open this notebook from the reproducibility folder or its notebooks subfolder.')
if 'dms' in sys.modules and Path(sys.modules['dms'].__file__).resolve().parent != PROJECT_ROOT / 'dms':
    raise RuntimeError('Restart the kernel to use the bundled dms package.')
sys.path.insert(0, str(PROJECT_ROOT))
for name in ('dms.config', 'dms.stieltjes', 'dms.simulation', 'dms.theory',
             'dms.jobs', 'dms.optimization', 'dms.ridge_diagnostics',
             'dms.exploration', 'dms.result_cache', 'dms.parameter_planes',
             'dms.infinity_diagnostics', 'dms.breadth_benchmark',
             'dms.breadth_optimum', 'dms.breadth_transitions'):
    importlib.reload(importlib.import_module(name))
from dms.exploration import SweepParameters
from dms.result_cache import ResultCache
from tools.portable_results import load_results, save_results, assert_settings_match

plt.rcParams.update({'axes.labelsize': 11, 'axes.labelweight': 'bold',
                     'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11})
FIGURES = {}


## Numerical controls

In [ ]:
# False redraws the published results; True recomputes the numerical stages.
RECOMPUTE = False
FORCE_COMPUTE = False

BREADTH3 = dict(
    base=SweepParameters(q=20, sigma=1., G2=100., f=.1, ftilde=.2,
        eta=.2, F=.4, FTest=.4, Fneq=.2, rhoneq=.25,
        m=.05, mneq=.1, mneqtilde=.1),
    Fneq_values=[.01, .1, .2, .3],
    r_values=[.3, 3.],
    fixed_ftilde=.1,
    f_grid=np.linspace(.001, .95, 101),
    ftilde_grid=np.linspace(.005, .2, 41),
    f_search=dict(bounds=(.001, .95), n_probe=11, tolerance=.005,
                  max_iter=24, extra_probes=[]),
    cap_resolution=dict(growth=100., max_extensions=2, bracket_steps=64,
                        risk_margin=1e-10),
    reuse_f_grid=np.linspace(.001, .95, 101),
    mean_convention='normalized', reference_L=100,
    search=dict(method='nonnegative', lambda_max=1000., n_grid=641),
)
BREADTH3_TRANSITION_OPTIONS = dict(barrier_rtol=1e-8, n_probe=0)
BREADTH3_CACHES = dict(
    risk=dict(directory=PROJECT_ROOT / '.cache' / 'breadth_risk', enabled=True, verbose=True),
    optimum=dict(directory=PROJECT_ROOT / '.cache' / 'breadth_optimum', enabled=True, verbose=True),
)
BREADTH3_TRANSITION_CACHE = dict(
    directory=PROJECT_ROOT / '.cache' / 'breadth_transitions', enabled=True, verbose=True)
DATA_ARCHIVE = PROJECT_ROOT / 'data' / 'library_breadth'


def numerical_settings():
    return dict(breadth=BREADTH3, transitions=BREADTH3_TRANSITION_OPTIONS)


## Load bundled results

In [ ]:
if RECOMPUTE:
    BREADTH_DATA, BREADTH_IDS = {}, {}
else:
    BREADTH_DATA, metadata = load_results(DATA_ARCHIVE, restore_records=True)
    assert_settings_match(metadata['settings'], numerical_settings())
    BREADTH_IDS = metadata['source_identities']
    print('Loaded bundled risk cuts, optimal breadths, and transition evidence.')


## Optional numerical stages

In [ ]:
def breadth_three_stage(stage, settings, *, compute=False, force=False):
    """One independent numerical stage, or strict cache-only restore."""
    import importlib
    import dms.infinity_diagnostics as tail_module
    import dms.breadth_benchmark as benchmark_module
    import dms.breadth_optimum as optimum_module
    importlib.reload(tail_module)
    benchmark_module = importlib.reload(benchmark_module)
    optimum_module = importlib.reload(optimum_module)
    if len(settings['r_values']) != 2 or len(set(settings['r_values'])) != 2:
        raise ValueError('Choose two distinct r values for the left and middle panels')
    stages = dict(risk_small=(0, 'risk'), risk_large=(1, 'risk'),
                  fopt_small=(0, 'optimum'), fopt_large=(1, 'optimum'))
    index, kind = stages[stage]
    r = float(settings['r_values'][index])
    common = {key: deepcopy(settings[key]) for key in
              ('base', 'Fneq_values', 'mean_convention', 'reference_L', 'search')}
    if kind == 'risk':
        case = dict(common, f_grid=np.asarray(settings['f_grid']),
                    fixed_ftilde=float(settings['fixed_ftilde']))
        data, identity = benchmark_module.cached_breadth_benchmark_case(
            case, r, cache=ResultCache(**BREADTH3_CACHES[kind]), compute=compute, force=force)
        if compute:
            for row in data['diagnostics']:
                print(f"{stage}, Fneq={row['Fneq']:g}: {row['status_counts']}")
    else:
        case = dict(common, r=r, **{key: deepcopy(settings[key]) for key in
                    ('ftilde_grid', 'f_search', 'cap_resolution', 'reuse_f_grid')})
        data, identity = optimum_module.cached_breadth_optimum_study(
            case, cache=ResultCache(**BREADTH3_CACHES[kind]), compute=compute, force=force)
        if compute:
            for value, rows in data['profiles']:
                print(f"{stage}, Fneq={value:g}: "
                      f"{sum(row['complete'] for row in rows)}/{len(rows)} resolved")
                for row in rows:
                    if not row['complete']:
                        print(f"  ftilde={row['ftilde']:g}: {row['status_counts']}; {row['reasons']}")
    return data, identity


def breadth_three_transitions(study, study_identity, *, compute=False, force=False):
    import importlib
    import dms.breadth_transitions as transition_module
    transition_module = importlib.reload(transition_module)
    return transition_module.cached_breadth_transition_evidence(
        study, study_identity, cache=ResultCache(**BREADTH3_TRANSITION_CACHE),
        point_cache=ResultCache(**dict(BREADTH3_CACHES['optimum'], verbose=False)),
        **BREADTH3_TRANSITION_OPTIONS, compute=compute, force=force)

In [ ]:
if RECOMPUTE:
    BREADTH_DATA['risk_small'], BREADTH_IDS['risk_small'] = breadth_three_stage(
        'risk_small', BREADTH3, compute=True, force=FORCE_COMPUTE)


In [ ]:
if RECOMPUTE:
    BREADTH_DATA['risk_large'], BREADTH_IDS['risk_large'] = breadth_three_stage(
        'risk_large', BREADTH3, compute=True, force=FORCE_COMPUTE)


In [ ]:
if RECOMPUTE:
    BREADTH_DATA['fopt_small'], BREADTH_IDS['fopt_small'] = breadth_three_stage(
        'fopt_small', BREADTH3, compute=True, force=FORCE_COMPUTE)


In [ ]:
if RECOMPUTE:
    BREADTH_DATA['fopt_large'], BREADTH_IDS['fopt_large'] = breadth_three_stage(
        'fopt_large', BREADTH3, compute=True, force=FORCE_COMPUTE)


In [ ]:
if RECOMPUTE:
    for suffix in ('small', 'large'):
        stage = 'transitions_' + suffix
        BREADTH_DATA[stage], BREADTH_IDS[stage] = breadth_three_transitions(
            BREADTH_DATA['fopt_' + suffix], BREADTH_IDS['fopt_' + suffix],
            compute=True, force=FORCE_COMPUTE)


## Figure construction

In [ ]:
def draw_breadth3_risk_axis(ax, case, colours, style, linestyle, *, inset=False, exclude_values=()):
    """Draw stored cuts; return sampled minima for choosing the inset window."""
    minima, series = [], []
    for value, data in case['curves']:
        if any(np.isclose(value, hidden, rtol=0, atol=1e-12) for hidden in exclude_values):
            continue
        x = np.asarray(data['x_values'])
        y = np.asarray(data['risk'][0] / data['sigma2'][0])
        finite = np.isfinite(y)
        ax.plot(x, np.where(finite, y, np.nan), color=colours[value],
                ls=linestyle, lw=style['linewidth'])
        series.append((x, y))
        if np.any(finite):
            j = int(np.argmin(np.where(finite, y, np.inf)))
            minima.append((float(x[j]), float(y[j])))
            if style['show_minima']:
                marker = style['edge_marker'] if j in (0, len(x)-1) else style['minimum_marker']
                ax.plot(x[j], y[j], ls='none', marker=marker, color=colours[value],
                    ms=style['inset']['marker_size'] if inset else style['marker_size'],
                    mfc=style['marker_facecolor'], mew=style['marker_edgewidth'], zorder=5)
        cap = finite & (data['status'][0] == 'upper')
        if style['show_cap_markers']:
            ax.plot(x[cap], y[cap], ls='none', marker=style['cap_marker'],
                    color=colours[value], mfc='none', ms=style['cap_marker_size'])
        isolated = finite & ~(np.r_[False, finite[:-1]] | np.r_[finite[1:], False])
        ax.plot(x[isolated], y[isolated], ls='none', marker=style['isolated_marker'],
                color=colours[value], ms=style['marker_size'])
    if style['show_analytic_reference']:
        ax.axvline(case['benchmark']['f_star'], **style['analytic_line'])
    ax.grid(False)
    return minima, series


def draw_breadth3_optimum_curve(ax, x, y, evidence, colour, linestyle, style, override=None):
    """Draw branches separately and connect only diagnosed/manual jump intervals."""
    mode = style.get('opt_transition_mode', 'dashed')
    if mode not in ('dashed', 'gap', 'solid'):
        raise ValueError('opt_transition_mode must be dashed, gap or solid')
    jump = np.asarray(evidence['jump_mask'], dtype=bool).copy()
    if jump.shape != (max(0, len(x)-1),):
        raise ValueError('Transition evidence does not match this curve')
    if override is not None:
        # Explicit display choices replace the automatic evidence for this curve.
        jump[:] = False
        for left, right in override:
            matches = np.flatnonzero(np.isclose(x[:-1], left, rtol=0, atol=1e-12)
                                    & np.isclose(x[1:], right, rtol=0, atol=1e-12))
            if len(matches) != 1:
                raise ValueError(f'Manual jump interval {(left, right)} must match adjacent test-breadth samples')
            jump[matches[0]] = True
    jump &= np.isfinite(y[:-1]) & np.isfinite(y[1:])
    if mode == 'solid':
        jump[:] = False
    # NaNs interrupt the ordinary path; no solid line remains underneath a dash.
    plot_x, plot_y = [], []
    for i, (xx, yy) in enumerate(zip(x, y)):
        plot_x.append(xx)
        plot_y.append(yy)
        if i < len(jump) and jump[i]:
            plot_x.append(np.nan)
            plot_y.append(np.nan)
    ax.plot(plot_x, plot_y, color=colour, ls=linestyle, lw=style['linewidth'])
    if mode == 'dashed':
        width = style.get('opt_jump_linewidth')
        if width is None:
            width = style['linewidth']
        for i in np.flatnonzero(jump):
            ax.plot(x[i:i+2], y[i:i+2], color=colour,
                    ls=style.get('opt_jump_linestyle', '--'), lw=width,
                    zorder=style.get('opt_jump_zorder', 3))


def draw_breadth_three_panel(data, style):
    """Editable three-panel renderer; reads cached numerical arrays only."""
    from matplotlib.ticker import MaxNLocator
    from matplotlib.patches import Rectangle
    cuts = [data['risk_small'], data['risk_large']]
    optima = [data['fopt_small'], data['fopt_large']]
    values = list(cuts[0]['metadata']['Fneq_values'])
    palette = style['colours']
    if palette is None:
        palette = [plt.get_cmap(style['cmap'])(v)
                   for v in np.linspace(*style['colour_range'], len(values))]
    if len(palette) < len(values):
        raise ValueError('Provide one colour for each Fneq value')
    colours = dict(zip(values, palette))
    if len(style['r_linestyles']) != 2:
        raise ValueError('Provide a linestyle for each of the two r values')
    fig, axes = plt.subplots(1, 3, figsize=(style['width'], style['height']),
        dpi=style['dpi'], layout='constrained',
        gridspec_kw=dict(width_ratios=style['width_ratios']))
    fig.get_layout_engine().set(w_pad=style['w_pad'], h_pad=style['h_pad'],
                               wspace=style['wspace'])
    cut_contents = []
    for col, (ax, case) in enumerate(zip(axes[:2], cuts)):
        cut_contents.append(draw_breadth3_risk_axis(
            ax, case, colours, style, style['r_linestyles'][col],
            exclude_values=style.get('left_exclude_Fneq', []) if col == 0 else ()))
        meta = case['metadata']
        ax.set_title(style['risk_title'].format(r=meta['r'], ftilde=meta['fixed_ftilde']),
                     fontsize=style['title_size'], pad=style['title_pad'])
        ax.set_xlabel(style['risk_xlabel'], fontsize=style['label_size'])
        if col == 0:
            ax.set_ylabel(style['risk_ylabel'], fontsize=style['label_size'])
        panel = ('left', 'middle')[col]
        ax.set_yscale(style[f'{panel}_yscale'])
        if style[f'{panel}_xticks'] is not None: ax.set_xticks(style[f'{panel}_xticks'])
        if style[f'{panel}_yticks'] is not None: ax.set_yticks(style[f'{panel}_yticks'])
        if style[f'{panel}_xlim'] is not None: ax.set_xlim(style[f'{panel}_xlim'])
        if style[f'{panel}_ylim'] is not None: ax.set_ylim(style[f'{panel}_ylim'])
    # Inset: same low-r data, colours and dashed style; no legend.
    zoom = style['inset']
    if zoom['show']:
        inset_ax = axes[0].inset_axes(zoom['bounds'])
        draw_breadth3_risk_axis(inset_ax, cuts[0], colours, style,
                              style['r_linestyles'][0], inset=True,
                              exclude_values=style.get('left_exclude_Fneq', []))
        minima, series = cut_contents[0]
        xlim = zoom['xlim']
        if xlim is None:
            centers = [x for x, _ in minima] + [cuts[0]['benchmark']['f_star']]
            low, high = min(centers), max(centers)
            span = max(high-low, zoom['minimum_x_span'])
            center = (low+high)/2
            grid = cuts[0]['metadata']['f_grid']
            xlim = (max(float(grid[0]), center-span*(.5+zoom['x_padding'])),
                    min(float(grid[-1]), center+span*(.5+zoom['x_padding'])))
        ylim = zoom['ylim']
        if ylim is None:
            pieces = [y[(x >= xlim[0]) & (x <= xlim[1]) & np.isfinite(y)] for x,y in series]
            visible = np.concatenate(pieces) if pieces else np.array([])
            if visible.size:
                lo, hi = float(visible.min()), float(visible.max())
                span = max(hi-lo, abs(hi)*.02, 1e-9)
                ylim = (lo-zoom['y_padding']*span, hi+zoom['y_padding']*span)
        inset_ax.set_xlim(xlim)
        if ylim is not None: inset_ax.set_ylim(ylim)
        inset_ax.xaxis.set_major_locator(MaxNLocator(zoom['x_nbins']))
        inset_ax.yaxis.set_major_locator(MaxNLocator(zoom['y_nbins']))
        if zoom['xticks'] is not None: inset_ax.set_xticks(zoom['xticks'])
        if zoom['yticks'] is not None: inset_ax.set_yticks(zoom['yticks'])
        inset_ax.tick_params(labelsize=zoom['tick_size'], length=zoom['tick_length'])
        inset_ax.set_title(zoom['title'], fontsize=zoom['title_size'])
        if zoom['show_connectors']:
            indicator = axes[0].indicate_inset_zoom(inset_ax, **zoom['connector_style'])
            # Keep the built-in rectangle transparent, not invisible:
            # Matplotlib can group same-style connectors into its draw path,
            # so hiding the rectangle would also hide those connectors.
            indicator_rectangle = (indicator.rectangle if hasattr(indicator, 'rectangle')
                                   else indicator[0])
            indicator_rectangle.set_alpha(0.)
        # Outline the actual zoom window in the parent axis's data coordinates.
        # Draw above curves/markers, independently of the faint connector lines.
        if zoom.get('show_region', True):
            x0, x1 = inset_ax.get_xlim()
            y0, y1 = inset_ax.get_ylim()
            axes[0].add_patch(Rectangle(
                (x0, y0), x1-x0, y1-y0, fill=False,
                **zoom.get('region_style', dict(edgecolor='.2', linewidth=1.,
                                               linestyle='-', alpha=1., zorder=6))))
    # Right panel: same colour mapping and r styles as the two risk cuts.
    for index, case in enumerate(optima):
        transitions = dict(data['transitions_small' if index == 0 else 'transitions_large']['curves'])
        for value, rows in case['profiles']:
            if any(np.isclose(value, hidden, rtol=0, atol=1e-12)
                   for hidden in style.get('opt_exclude_Fneq', [])):
                continue
            x = np.array([row['ftilde'] for row in rows])
            y = np.array([row['f_opt'] if row['complete'] else np.nan for row in rows])
            ax = axes[2]
            override = style.get('opt_jump_overrides', {}).get((float(case['metadata']['r']), float(value)))
            draw_breadth3_optimum_curve(ax, x, y, transitions[value], colours[value],
                style['r_linestyles'][index], style, override=override)
            finite = np.isfinite(y)
            isolated = finite & ~(np.r_[False, finite[:-1]] | np.r_[finite[1:], False])
            ax.plot(x[isolated], y[isolated], ls='none', marker=style['isolated_marker'],
                    color=colours[value], ms=style['marker_size'])
            if style['show_edge_markers']:
                edge = finite & np.array([row['at_boundary'] for row in rows])
                ax.plot(x[edge], y[edge], ls='none', marker=style['edge_marker'],
                        color=colours[value], ms=style['marker_size'], mfc=style['marker_facecolor'])
    if style['show_analytic_reference']:
        axes[2].plot(optima[0]['metadata']['ftilde_grid'], optima[0]['small_r_f_opt'],
                     **style['analytic_line'])
    axes[2].set_xlabel(style['opt_xlabel'], fontsize=style['label_size'])
    axes[2].set_ylabel(style['opt_ylabel'], fontsize=style['label_size'])
    axes[2].set_title(style['opt_title'], fontsize=style['title_size'], pad=style['title_pad'])
    if style['opt_xticks'] is not None: axes[2].set_xticks(style['opt_xticks'])
    if style['opt_yticks'] is not None: axes[2].set_yticks(style['opt_yticks'])
    if style['opt_xlim'] is not None: axes[2].set_xlim(style['opt_xlim'])
    if style['opt_ylim'] is not None: axes[2].set_ylim(style['opt_ylim'])
    # One colour legend, in b. The reference (and optional r-style keys) lives in c.
    if style['show_colour_legend']:
        handles = [Line2D([], [], color=colours[value], lw=style['linewidth'],
                   ls=style['r_linestyles'][1], label=style['Fneq_label'].format(value=value))
                   for value in values]
        axes[1].legend(handles=handles, fontsize=style['legend_size'], **style['colour_legend'])
    right_handles = []
    if style['right_legend_include_r']:
        right_handles.extend(Line2D([], [], color=style['r_legend_colour'],
            ls=style['r_linestyles'][i], lw=style['linewidth'],
            label=style['r_label'].format(r=case['metadata']['r'])) for i,case in enumerate(optima))
    if style['show_analytic_reference']:
        right_handles.append(Line2D([], [], **style['analytic_line'], label=style['analytic_label']))
    if style['show_reference_legend'] and right_handles:
        axes[2].legend(handles=right_handles, fontsize=style['legend_size'], **style['reference_legend'])
    for i,ax in enumerate(axes):
        ax.tick_params(labelsize=style['tick_size'])
        ax.grid(False)
        ax.text(*style['panel_position'], style['panel_labels'][i], transform=ax.transAxes,
                fontsize=style['panel_size'], fontweight='bold')
    return fig

## Presentation controls

In [ ]:
# Presentation only: changing these controls does not recompute risk.
BREADTH3_STYLE = {'width': 10,
 'height': 3,
 'dpi': 140,
 'width_ratios': [1, 1, 1],
 'w_pad': 0.015,
 'h_pad': 0.035,
 'wspace': 0.06,
 'cmap': 'cividis',
 'colour_range': [0.15, 0.85],
 'colours': None,
 'linewidth': 1.8,
 'r_linestyles': ['-.', '-'],
 'show_minima': True,
 'minimum_marker': 'o',
 'marker_size': 6,
 'marker_facecolor': 'white',
 'marker_edgewidth': 1.1,
 'show_edge_markers': False,
 'edge_marker': 'v',
 'isolated_marker': 'o',
 'show_cap_markers': False,
 'cap_marker': '^',
 'cap_marker_size': 3.0,
 'show_analytic_reference': True,
 'analytic_line': {'color': 'black', 'ls': '--', 'lw': 1.8, 'zorder': 3},
 'analytic_label': '$f_*^{(0)}$',
 'risk_xlabel': 'Training mutation rate $f$',
 'risk_ylabel': 'Optimal risk $R_{\\mathrm{opt}}/\\sigma^2$',
 'risk_title': '$r={r:g},\\ \\tilde f={ftilde:g}$',
 'left_exclude_Fneq': [],
 'left_xlim': [0.0, 0.95],
 'left_xticks': [0.1, 0.3, 0.5, 0.7, 0.9],
 'left_ylim': [0, 1],
 'left_yticks': None,
 'left_yscale': 'linear',
 'middle_xlim': [0.0, 0.95],
 'middle_xticks': [0.1, 0.3, 0.5, 0.7, 0.9],
 'middle_ylim': [0.8, 5.9],
 'middle_yticks': None,
 'middle_yscale': 'linear',
 'opt_exclude_Fneq': [],
 'opt_transition_mode': 'dashed',
 'opt_jump_linestyle': '--',
 'opt_jump_linewidth': None,
 'opt_jump_zorder': 3,
 'opt_jump_overrides': {},
 'opt_xlabel': 'Test mutation rate $\\tilde f$',
 'opt_ylabel': 'Optimal training mut. rate $f_{\\mathrm{opt}}$',
 'opt_title': '',
 'opt_xlim': None,
 'opt_ylim': [0, 0.8],
 'opt_xticks': None,
 'opt_yticks': None,
 'label_size': 11,
 'tick_size': 11,
 'title_size': 11,
 'title_pad': 8.0,
 'legend_size': 11,
 'Fneq_label': '$F_{{\\neq}}={value:g}$',
 'r_label': '$r={r:g}$',
 'show_colour_legend': True,
 'colour_legend': {'loc': 'upper right', 'frameon': True, 'handlelength': 3},
 'show_reference_legend': True,
 'reference_legend': {'loc': [0.58, 0.54], 'frameon': True, 'handlelength': 3},
 'right_legend_include_r': True,
 'r_legend_colour': '.35',
 'panel_labels': ['a', 'b', 'c'],
 'panel_position': [-0.02, 1.04],
 'panel_size': 12,
 'inset': {'show': True,
           'bounds': [0.35, 0.49, 0.56, 0.43],
           'xlim': None,
           'ylim': None,
           'minimum_x_span': 0.08,
           'x_padding': 0.2,
           'y_padding': 0.12,
           'xticks': None,
           'yticks': [],
           'x_nbins': 3,
           'y_nbins': 3,
           'tick_size': 8,
           'tick_length': 2.5,
           'marker_size': 6,
           'title': '',
           'title_size': 9,
           'show_connectors': True,
           'connector_style': {'edgecolor': '.5', 'linewidth': 0.8, 'alpha': 0.6},
           'show_region': False,
           'region_style': {'edgecolor': '.2',
                            'linewidth': 1.0,
                            'linestyle': '-',
                            'alpha': 1.0,
                            'zorder': 6}}}


In [ ]:
fig = draw_breadth_three_panel(BREADTH_DATA, BREADTH3_STYLE)
FIGURES['breadth_risk_and_optimal_breadth'] = fig
display(fig)


## Export the displayed figure

In [ ]:
PDF_FIGURES = {'breadth_risk_and_optimal_breadth': 'library_breadth_with_opt.pdf'}
OUTPUT_DIR = PROJECT_ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)
for key, filename in PDF_FIGURES.items():
    FIGURES[key].savefig(OUTPUT_DIR / filename, format='pdf', bbox_inches='tight')
    print('output/' + filename)


## Optional numerical export

In [ ]:
EXPORT_NUMERICAL = False
if EXPORT_NUMERICAL:
    save_results(OUTPUT_DIR / 'library_breadth_arrays', BREADTH_DATA,
                 metadata={'settings': numerical_settings(), 'source_identities': BREADTH_IDS})
